# HDD 12 - Recorded-failure evaluation on frozen September development scores

This is the first real-failure evaluation for the **Q3 acceptance group**, which has already served as development acceptance data. It is not an untouched final test, and it must not be used to retune thresholds, choose another candidate or claim final research performance.

No model fitting or inference occurs here. Before labels are opened, all 16 saved score tables (two encoders × four detectors × two threshold policies) are hashed and the evaluation protocol is saved. The failure column is extracted into a separate evaluation-only table from the same Q3 archive. No failure labels enter model inputs, calibration, clustering or adaptation.

Use the ZIP path that worked in Notebook 09. June stays closed. With only September endpoints, 30-day warning-event evaluation may have no complete pre-failure intervals; the notebook reports insufficient evidence rather than inventing a recall value.

In [1]:
from pathlib import Path
import json,hashlib,zipfile,re
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from IPython.display import display
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
EXPANSION_ID='q3_development_expansion_v1'
TRIAL_ID='offline_replay_adaptation_v1'
EVAL_ID='q3_recorded_failure_evaluation_v1'
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
TRIAL=COMPONENT_ROOT/'results'/RUN_ID/TRIAL_ID
EXP=COMPONENT_ROOT/'results'/RUN_ID/EXPANSION_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/EVAL_ID
LABEL_OUT=COMPONENT_ROOT/'data/evaluation_only'/RUN_ID/EVAL_ID
for p in [OUT,LABEL_OUT]:p.mkdir(parents=True,exist_ok=True)
INPUTS={}
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def remember(p):
    assert p.is_file(),f'Missing input: {p}'
    INPUTS[str(p.relative_to(COMPONENT_ROOT)) if p.is_relative_to(COMPONENT_ROOT) else str(p)]=sha256(p)
    return p
def load_json(p):return json.loads(remember(p).read_text())
trial=load_json(TRIAL/'offline_adaptation_complete.json')
assert trial['analysis_complete'] and trial['rollback_exercised']
assert not trial['sealed_test_opened'] and not trial['failure_labels_used']
assert trial['active_in_memory_model']=='frozen' and trial['deployed_model_updates']==0
exp=load_json(EXP/'expansion_complete.json')
splits=load_json(EXP/'q3_serials.json')
assert sha256(EXP/'q3_serials.json')==exp['split_sha256']
ACCEPTANCE_IDS=set(splits['acceptance'])
assert ACCEPTANCE_IDS.isdisjoint(splits['adaptation'])
assert Q3_ZIP.is_file(),f'Point Q3_ZIP to the ZIP used in Notebook 09: {Q3_ZIP}'
assert sha256(remember(Q3_ZIP))==exp['settings']['archive_sha256']
print('Notebook 11 decision:',trial['decision'])
print('Evaluation population:',len(ACCEPTANCE_IDS),'reserved development acceptance drives')

Notebook 11 decision: passed_pilot_checks_offline_review_only
Evaluation population: 100 reserved development acceptance drives


## 1. Freeze score tables and evaluation definitions before extracting labels

For endpoint `t` and horizon `h`, a positive target is a first recorded failure in `(t, t+h]`. The failure day itself is excluded from warning prediction, as are all later endpoints. A negative target requires an observed daily record on **every** day `t+1` through `t+h`, without a recorded failure in that interval. Missing follow-up and the quarter boundary are censored, not labelled healthy.

Window average precision is computed only when both target classes exist. The metric named `average_precision` is the standard non-interpolated AP summary of the precision-recall curve; it is not trapezoidal PR-AUC. Overlapping endpoints remain dependent; these descriptive pilot estimates do not have independent-window confidence intervals.

Strict event recall requires scored endpoints on every one of the `h` days before a failure. Partial-history events are reported separately and excluded from the strict recall denominator. No post-failure or same-day alerts count as warnings.

In [2]:
def bool_column(s):
    if s.dtype==bool:return s
    v=s.astype(str).str.lower();assert v.isin(['true','false']).all()
    return v.eq('true')
TABLES={};SCORE_HASHES={};reference=None
for model in ['frozen','candidate']:
    for detector in ['R','E','D','RD']:
        for policy in ['global','conditional']:
            p=remember(TRIAL/f'acceptance_{model}_{detector}_{policy}_scores.csv')
            SCORE_HASHES[p.name]=sha256(p)
            table=pd.read_csv(p,dtype={'serial_number':str},parse_dates=['start_date','end_date'])
            table['is_anomaly_candidate']=bool_column(table.is_anomaly_candidate)
            assert not table.duplicated(['serial_number','end_date']).any()
            assert set(table.serial_number).issubset(ACCEPTANCE_IDS)
            assert pd.to_datetime(table.end_date).between('2021-09-01','2021-09-30').all()
            assert np.isfinite(table[['score','threshold']].to_numpy()).all()
            np.testing.assert_array_equal(table.is_anomaly_candidate,table.score>table.threshold)
            keys=table[['serial_number','start_date','end_date']].reset_index(drop=True)
            if reference is None:reference=keys
            else:pd.testing.assert_frame_equal(keys,reference)
            TABLES[(model,detector,policy)]=table
SETTINGS={'horizons_days':[7,14,30],'positive_interval':'0 < failure_date - endpoint <= horizon',
    'negative_rule':'all next h calendar days observed; no failure within horizon',
    'event_rule':'all h pre-failure calendar endpoints must have valid saved scores',
    'failure_event':'first recorded failure per acceptance serial in Q3',
    'population':'already inspected September development acceptance group',
    'score_sha256':SCORE_HASHES,'archive_sha256':sha256(Q3_ZIP),
    'upstream_trial_complete_sha256':sha256(TRIAL/'offline_adaptation_complete.json'),
    'no_fitting_or_threshold_changes':True,'winner_selection_enabled':False}
p=OUT/'evaluation_protocol.json'
if p.exists():assert json.loads(p.read_text())==SETTINGS,'Changed protocol/scores: preserve the previous evaluation; do not overwrite it.'
p.write_text(json.dumps(SETTINGS,indent=2))
print('All',len(TABLES),'score tables frozen before failure extraction.')

All 16 score tables frozen before failure extraction.


## 2. Extract only dates, serials and recorded failures

The label reader does not load SMART features. It validates one binary failure value per observed drive-date. All available Q3 label records for the declared acceptance serials are retained, including dates before September, so an already failed drive is never relabelled negative later.

In [3]:
with zipfile.ZipFile(Q3_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if not match:continue
        date=match.group(1)
        if not ('2021-07-01'<=date<='2021-09-30'):continue
        assert date not in members,f'Duplicate daily member: {date}'
        members[date]=info.filename
    expected=set(pd.date_range('2021-07-01','2021-09-30').strftime('%Y-%m-%d'))
    assert set(members)==expected,'Q3 archive dates changed or incomplete.'
    parts=[]
    for date,member in sorted(members.items()):
        with z.open(member) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number','failure'],dtype={'serial_number':'string'},chunksize=50000):
                keep=chunk.loc[chunk.serial_number.isin(ACCEPTANCE_IDS)].copy()
                if keep.empty:continue
                keep['date']=pd.to_datetime(keep.date,errors='raise').dt.normalize()
                assert keep.date.eq(pd.Timestamp(date)).all(),'Filename/date mismatch.'
                keep['failure']=pd.to_numeric(keep.failure,errors='raise')
                assert keep.failure.notna().all() and keep.failure.isin([0,1]).all(),'Missing/nonbinary failure record.'
                parts.append(keep)
        if date.endswith(('01','15')):print('Read evaluation labels:',date)
labels=pd.concat(parts,ignore_index=True).sort_values(['serial_number','date']).reset_index(drop=True)
assert not labels.duplicated(['serial_number','date']).any()
assert set(labels.columns)=={'date','serial_number','failure'}
labels.to_parquet(LABEL_OUT/'recorded_failures_daily.parquet',index=False)
events=labels.loc[labels.failure.eq(1)].groupby('serial_number').date.min().rename('failure_date').reset_index()
events.to_csv(OUT/'first_recorded_failure_events.csv',index=False)
print('Recorded Q3 failure drives in the declared acceptance group:',len(events))
print('Recorded September failure drives:',int(events.failure_date.between('2021-09-01','2021-09-30').sum()))

Read evaluation labels: 2021-07-01
Read evaluation labels: 2021-07-15
Read evaluation labels: 2021-08-01
Read evaluation labels: 2021-08-15
Read evaluation labels: 2021-09-01
Read evaluation labels: 2021-09-15
Recorded Q3 failure drives in the declared acceptance group: 0
Recorded September failure drives: 0


## 3. Build endpoint targets with explicit censoring

Only exact serial/date joins are used. Follow-up may be known for a positive event even when intervening records are missing; a negative needs complete follow-up. This asymmetry is reported because the source records failure events, not continuous survival certification.

In [4]:
def targets_for(keys,daily_labels,failure_events,horizon):
    observed={s:set(g.date) for s,g in daily_labels.groupby('serial_number')}
    first=failure_events.set_index('serial_number').failure_date.to_dict()
    rows=[]
    for row in keys.itertuples(index=False):
        endpoint=pd.Timestamp(row.end_date);failure=first.get(row.serial_number)
        if failure is not None and failure<=endpoint:
            status='same_day_or_after_recorded_failure';target=None
        elif failure is not None and 0<(failure-endpoint).days<=horizon:
            status='observed_failure_within_horizon';target=1
        else:
            followup=pd.date_range(start=endpoint+pd.Timedelta(days=1),periods=horizon,freq='D')
            complete=all(d in observed.get(row.serial_number,set()) for d in followup)
            status='complete_observed_negative' if complete else 'censored_incomplete_followup'
            target=0 if complete else None
        rows.append({'serial_number':row.serial_number,'end_date':endpoint,'horizon_days':horizon,
            'target':target,'status':status,'failure_date':failure})
    return pd.DataFrame(rows)
TARGETS={};support=[]
for h in SETTINGS['horizons_days']:
    t=targets_for(reference,labels,events,h);TARGETS[h]=t
    t.to_csv(LABEL_OUT/f'endpoint_targets_h{h}.csv',index=False)
    support.append({'horizon_days':h,'scoreable_endpoints':len(t),'positive_endpoints':int(t.target.eq(1).sum()),
        'negative_endpoints':int(t.target.eq(0).sum()),'censored_endpoints':int(t.status.eq('censored_incomplete_followup').sum()),
        'same_day_or_postfailure_excluded':int(t.status.eq('same_day_or_after_recorded_failure').sum()),
        'drives_with_positive_endpoints':t.loc[t.target.eq(1),'serial_number'].nunique(),
        'both_endpoint_classes_available':bool(t.target.eq(0).any() and t.target.eq(1).any())})
support=pd.DataFrame(support);support.to_csv(OUT/'failure_evaluation_support.csv',index=False)
display(support)

C:\Users\IMAKA\AppData\Local\Temp\ipykernel_7556\673801193.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  followup=pd.date_range(start=endpoint+pd.Timedelta(days=1),periods=horizon,freq='D')


,horizon_days,scoreable_endpoints,positive_endpoints,negative_endpoints,censored_endpoints,same_day_or_postfailure_excluded,drives_with_positive_endpoints,both_endpoint_classes_available
0,7,2989,0,2289,700,0,0,False
1,14,2989,0,1589,1400,0,0,False
2,30,2989,0,0,2989,0,0,False


## 4. Evaluate all saved alternatives without selecting a winner

False alerts per 1,000 **observed negative drive-endpoints** use horizon-specific known negatives as the denominator. They are not fleet-wide false alerts per 1,000 operating drive-days. Censored endpoints cannot enter that denominator. Precision refers to the chosen future-failure horizon, not to whether an alert indicates any other physical anomaly.

In [5]:
def endpoint_metrics(table,targets):
    valid=targets.target.notna().to_numpy();y=targets.loc[valid,'target'].to_numpy(dtype=int)
    score=table.loc[valid,'score'].to_numpy();flag=table.loc[valid,'is_anomaly_candidate'].to_numpy(dtype=bool)
    positives=int((y==1).sum());negatives=int((y==0).sum())
    tp=int((flag&(y==1)).sum());fp=int((flag&(y==0)).sum())
    return {'evaluated_endpoints':len(y),'positive_endpoints':positives,'negative_endpoints':negatives,
        'average_precision':float(average_precision_score(y,score)) if positives and negatives else np.nan,
        'positive_prevalence':float(y.mean()) if len(y) else np.nan,
        'threshold_precision':tp/(tp+fp) if tp+fp else np.nan,
        'threshold_recall':tp/positives if positives else np.nan,
        'true_positive_endpoints':tp,'false_positive_endpoints':fp,
        'false_alerts_per_1000_observed_negative_endpoints':1000*fp/negatives if negatives else np.nan,
        'status':'both_classes_present' if positives and negatives else 'insufficient_endpoint_classes'}

def event_metrics(table,failure_events,horizon):
    details=[]
    for event in failure_events.itertuples(index=False):
        if not (pd.Timestamp('2021-09-01')<=event.failure_date<=pd.Timestamp('2021-09-30')):continue
        required=pd.date_range(end=event.failure_date-pd.Timedelta(days=1),periods=horizon,freq='D')
        g=table.loc[table.serial_number.eq(event.serial_number)&table.end_date.isin(required)]
        full=bool(len(g)==horizon)
        alerted=g.loc[g.is_anomaly_candidate]
        recalled=bool(full and not alerted.empty)
        lead=(event.failure_date-alerted.end_date.min()).days if recalled else np.nan
        details.append({'serial_number':event.serial_number,'failure_date':event.failure_date,
            'horizon_days':horizon,'available_pre_failure_endpoints':len(g),'full_warning_history':full,
            'any_alert_in_available_history':bool(not alerted.empty),'strict_event_recalled':recalled,
            'first_warning_lead_days':lead})
    d=pd.DataFrame(details,columns=['serial_number','failure_date','horizon_days','available_pre_failure_endpoints','full_warning_history','any_alert_in_available_history','strict_event_recalled','first_warning_lead_days'])
    full=d.loc[d.full_warning_history.eq(True)];recalled=int(full.strict_event_recalled.sum())
    return {'recorded_September_events':len(d),'events_with_complete_warning_history':len(full),
        'events_excluded_for_partial_history':len(d)-len(full),'strict_events_recalled':recalled,
        'strict_event_recall':recalled/len(full) if len(full) else np.nan,
        'median_first_warning_lead_days':float(full.first_warning_lead_days.median()) if recalled else np.nan},d
metrics=[];event_details=[]
for (model,detector,policy),table in TABLES.items():
    for h,t in TARGETS.items():
        # Tables and targets share the exact frozen row order validated above.
        pd.testing.assert_frame_equal(table[['serial_number','end_date']].reset_index(drop=True),t[['serial_number','end_date']].reset_index(drop=True),check_dtype=False)
        row={'model':model,'detector':detector,'policy':policy,'horizon_days':h}
        row.update(endpoint_metrics(table,t));er,ed=event_metrics(table,events,h);row.update(er)
        metrics.append(row);ed['model']=model;ed['detector']=detector;ed['policy']=policy;event_details.append(ed)
metrics=pd.DataFrame(metrics);metrics.to_csv(OUT/'recorded_failure_metrics.csv',index=False)
pd.concat(event_details,ignore_index=True).to_csv(OUT/'event_warning_details.csv',index=False)
display(metrics[['model','detector','policy','horizon_days','positive_endpoints','average_precision','events_with_complete_warning_history','strict_event_recall','status']])
print('Missing metric values indicate an undefined or unsupported estimate; they are not zeros.')

,model,detector,policy,horizon_days,positive_endpoints,average_precision,events_with_complete_warning_history,strict_event_recall,status
0,frozen,R,global,7,0,NaN,0,NaN,insufficient_endpoint_classes
1,frozen,R,global,14,0,NaN,0,NaN,insufficient_endpoint_classes
2,frozen,R,global,30,0,NaN,0,NaN,insufficient_endpoint_classes
3,frozen,R,conditional,7,0,NaN,0,NaN,insufficient_endpoint_classes
4,frozen,R,conditional,14,0,NaN,0,NaN,insufficient_endpoint_classes
5,frozen,R,conditional,30,0,NaN,0,NaN,insufficient_endpoint_classes
6,frozen,E,global,7,0,NaN,0,NaN,insufficient_endpoint_classes
7,frozen,E,global,14,0,NaN,0,NaN,insufficient_endpoint_classes
8,frozen,E,global,30,0,NaN,0,NaN,insufficient_endpoint_classes
9,frozen,E,conditional,7,0,NaN,0,NaN,insufficient_endpoint_classes


Missing metric values indicate an undefined or unsupported estimate; they are not zeros.


## 5. Verify labelling edge cases and preserve the evaluation record

This stage can validly finish with zero recorded failures or no evaluable 30-day warning events. In that case the correct next step is a broader event-coverage audit and a newly reserved future test protocol, not selecting failures to make this acceptance sample look stronger. Multi-model, multiple-seed and stronger baseline evaluation remain outstanding.

In [6]:
# Meaningful boundary checks independent of the real research outcomes.
toy_days=pd.date_range('2021-09-01','2021-09-30')
toy=pd.DataFrame({'serial_number':'toy','date':toy_days,'failure':0})
toy_events=pd.DataFrame({'serial_number':['toy'],'failure_date':[pd.Timestamp('2021-09-10')]})
toy_keys=pd.DataFrame({'serial_number':['toy']*3,'end_date':pd.to_datetime(['2021-09-03','2021-09-10','2021-09-29'])})
test=targets_for(toy_keys,toy,toy_events,7)
assert test.iloc[0].target==1 # Failure exactly h days later is included.
assert pd.isna(test.iloc[1].target) # Same-day failure cannot be an advance warning.
empty_events=toy_events.iloc[:0]
test=targets_for(toy_keys,toy,empty_events,7)
assert test.iloc[0].target==0 and pd.isna(test.iloc[2].target)
missing_day=toy.loc[toy.date.ne(pd.Timestamp('2021-09-05'))]
assert pd.isna(targets_for(toy_keys.iloc[:1],missing_day,empty_events,7).iloc[0].target)
# No failures -> AP/recall undefined, not a perfect score.
toy_score=pd.DataFrame({'score':[.1,.2,.3],'is_anomaly_candidate':[False,False,False]})
neg_targets=pd.DataFrame({'target':[0,0,0]})
assert np.isnan(endpoint_metrics(toy_score,neg_targets)['average_precision'])
assert np.isnan(endpoint_metrics(toy_score,neg_targets)['threshold_recall'])
for location,digest in INPUTS.items():
    p=Path(location) if Path(location).is_absolute() else COMPONENT_ROOT/location
    assert sha256(p)==digest,f'Input changed during evaluation: {p}'
result={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,'run_id':RUN_ID,
    'evaluation_id':EVAL_ID,'protocol':SETTINGS,'input_sha256':INPUTS,
    'recorded_failure_drives_Q3':len(events),'real_failure_labels_used_for_evaluation':True,
    'failure_labels_used_for_training_or_selection':False,'sealed_June_test_opened':False,
    'models_updated':0,'thresholds_changed':False,'winner_selected':False,
    'evaluation_scope':'previously inspected Q3 development acceptance scores',
    'limitations':['one model/seed','limited event support possible','right censoring at quarter end',
        'overlapping endpoints are dependent','strict event history excludes partial coverage',
        'not a blind final evaluation','source daily records do not certify continuous physical health']}
(OUT/'recorded_failure_evaluation_complete.json').write_text(json.dumps(result,indent=2,allow_nan=False))
print('NOTEBOOK 12 COMPLETE — recorded-failure development evaluation saved.')
print('Recorded Q3 failure drives:',len(events))
# print('Share this executed notebook, failure_evaluation_support.csv and recorded_failure_metrics.csv.')
print('No fitting, threshold adjustment or winner selection performed. June stays closed.')
print('Results:',OUT)

C:\Users\IMAKA\AppData\Local\Temp\ipykernel_7556\673801193.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  followup=pd.date_range(start=endpoint+pd.Timedelta(days=1),periods=horizon,freq='D')
C:\Users\IMAKA\AppData\Local\Temp\ipykernel_7556\673801193.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  followup=pd.date_range(start=endpoint+pd.Timedelta(days=1),periods=horizon,freq='D')


NOTEBOOK 12 COMPLETE — recorded-failure development evaluation saved.
Recorded Q3 failure drives: 0
No fitting, threshold adjustment or winner selection performed. June stays closed.
Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_recorded_failure_evaluation_v1
